# Exercice 13b : apprentissage par transfert (transfer learning) avec ResNet-50 (Colab, GPU T4)

**Chapitre lié** : 13. Réseaux convolutifs (CNN).

**Objectif.** Mesurer ce que rapporte un réseau **pré-entraîné** sur ImageNet quand on n'a
que 3 680 images étiquetées : classer les 37 races de chats et de chiens du jeu
Oxford-IIIT Pet avec le même ResNet-50 entraîné de trois façons.

| Expérience | Poids de départ | Ce qui est entraîné | Époques |
|---|---|---|---|
| A. De zéro | aléatoires | tout le réseau (23,6 M paramètres) | 10 |
| B. Extraction de caractéristiques | ImageNet (`IMAGENET1K_V2`) | la dernière couche seule (75 813 paramètres) | 5 |
| C. Réglage fin (fine-tuning) | ImageNet (`IMAGENET1K_V2`) | tout le réseau, taux 10 fois plus faible pour le tronc | 5 |

**Perte optimisée** : l'entropie croisée multiclasse (37 classes),
$\mathcal{L} = -\frac1N\sum_i \log \hat p_{i,y_i}$.

**Avant de lancer** : Exécution › Modifier le type d'exécution › **GPU T4**, puis
Exécution › Tout exécuter. Durée estimée : 25 à 35 minutes (non mesurée sur T4 au
moment de l'écriture). Coût : 0 €.

In [ ]:
# 1. Vérification du GPU : ce notebook est écrit pour un GPU NVIDIA T4 (Colab gratuit, ou Kaggle « GPU T4 x2 »)
!nvidia-smi
import torch
print("PyTorch", torch.__version__, "| CUDA disponible :", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("ATTENTION : pas de GPU. Exécution › Modifier le type d'exécution › GPU T4, puis tout relancer.\n"
          "Sur CPU, l'expérience A prendrait plusieurs heures.")

In [ ]:
# 2. Dépendances : torch et torchvision sont déjà dans l'image Colab (on ne les épingle pas,
# cela réinstallerait ~1 Go). Seul `datasets` est épinglé (version vérifiée sur PyPI, Python 3.12).
!pip install -q datasets==5.1.0

In [ ]:
# 3. Imports et configuration
import json
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from torch import nn
from torchvision import models
from torchvision.transforms import v2

# IA_TEST_LOCAL=1 : petite version (64 images, 1 époque) pour vérifier le code hors Colab
TEST_LOCAL = os.environ.get("IA_TEST_LOCAL") == "1"
SEED = 42
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
FP16 = device == "cuda"   # demi-précision float16 (le T4 ne fait pas de bfloat16)

def seed_everything(graine=SEED):
    random.seed(graine)
    np.random.seed(graine)
    torch.manual_seed(graine)
    torch.cuda.manual_seed_all(graine)

seed_everything()
config = dict(
    n_images=64 if TEST_LOCAL else None,       # None = tout le jeu (3 680 + 3 669 images)
    epoques_zero=1 if TEST_LOCAL else 10,
    epoques_gel=1 if TEST_LOCAL else 5,
    epoques_fine=1 if TEST_LOCAL else 5,
    lot=32 if TEST_LOCAL else 64,
    workers=0 if TEST_LOCAL else 2,            # Colab gratuit : 2 cœurs CPU
    taille=224,                                 # taille des images vues par ResNet-50
    lr_zero=1e-3, lr_tete=1e-3, lr_tronc=1e-4,
    weight_decay=1e-4,
)
print("Périphérique :", device, "| fp16 :", FP16, "| test local :", TEST_LOCAL)
print(config)

## Données : Oxford-IIIT Pet

37 races (12 de chats, 25 de chiens), environ 100 images par race pour l'entraînement et
autant pour le test. Jeu publié par Parkhi et al. (2012) ; copie Hugging Face
`timm/oxford-iiit-pet` (licence CC BY-SA 4.0, aucune demande d'accès, ~790 Mo).

**Augmentation** (entraînement seulement), tirée **image par image** dans `__getitem__` :
recadrage aléatoire redimensionné à 224×224 (`RandomResizedCrop`) et retournement
horizontal. En test : redimensionnement à 256 puis découpe centrale 224×224. Les deux
normalisent avec la moyenne et l'écart type d'ImageNet, comme lors du pré-entraînement.

In [ ]:
# 4. Chargement du jeu de données (téléchargé une fois dans le cache Hugging Face)
pets = load_dataset("timm/oxford-iiit-pet")
brut_train, brut_test = pets["train"], pets["test"]
if config["n_images"]:
    brut_train = brut_train.select(range(config["n_images"]))
    brut_test = brut_test.select(range(config["n_images"]))
CLASSES = brut_train.features["label"].names
print(f"{len(brut_train)} images d'entraînement, {len(brut_test)} de test, {len(CLASSES)} classes")

MOYENNE, ECART_TYPE = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)   # ImageNet
aug_train = v2.Compose([
    v2.ToImage(),
    v2.RandomResizedCrop(config["taille"], scale=(0.35, 1.0), antialias=True),
    v2.RandomHorizontalFlip(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MOYENNE, ECART_TYPE),
])
trans_test = v2.Compose([
    v2.ToImage(),
    v2.Resize(256, antialias=True),
    v2.CenterCrop(config["taille"]),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MOYENNE, ECART_TYPE),
])

class Pets(torch.utils.data.Dataset):
    # Transformation appliquée à UNE image à la fois : paramètres aléatoires tirés image par image
    def __init__(self, brut, transform):
        self.brut, self.transform = brut, transform
    def __len__(self):
        return len(self.brut)
    def __getitem__(self, i):
        ex = self.brut[i]
        return self.transform(ex["image"].convert("RGB")), ex["label"]

def chargeur(brut, transform, melanger):
    return torch.utils.data.DataLoader(
        Pets(brut, transform), batch_size=config["lot"], shuffle=melanger,
        num_workers=config["workers"], pin_memory=(device == "cuda"),
        persistent_workers=config["workers"] > 0,
        generator=torch.Generator().manual_seed(SEED))

train_dl = chargeur(brut_train, aug_train, True)
test_dl = chargeur(brut_test, trans_test, False)

## Boucle d'entraînement commune

AdamW (chapitre 02), taux d'apprentissage en cosinus, entropie croisée, calcul en float16
avec mise à l'échelle de la perte (`GradScaler`) pour éviter que les petits gradients
s'arrondissent à zéro. Exactitude mesurée sur les 3 669 images de test à chaque époque.

In [ ]:
# 5. Entraînement et évaluation
def evaluer(modele):
    modele.eval()
    bons, n = 0, 0
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=FP16):
        for x, y in test_dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            bons += (modele(x).argmax(1) == y).sum().item()
            n += len(y)
    return bons / n

def entrainer(nom, modele, groupes, epoques, geler_bn=False):
    # groupes : liste de {"params": ..., "lr": ...} (taux différent pour le tronc et la tête)
    modele = modele.to(device)
    optim = torch.optim.AdamW(groupes, weight_decay=config["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epoques * len(train_dl))
    scaler = torch.amp.GradScaler("cuda", enabled=FP16)
    hist = {"perte": [], "prec_test": [], "duree": []}
    t0 = time.perf_counter()
    for ep in range(epoques):
        modele.train()
        if geler_bn:   # tronc gelé : ses BatchNorm gardent les statistiques d'ImageNet
            modele.apply(lambda m: m.eval() if isinstance(m, nn.BatchNorm2d) else None)
        somme, n = 0.0, 0
        for x, y in train_dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=FP16):
                perte = F.cross_entropy(modele(x), y)
            optim.zero_grad(set_to_none=True)
            scaler.scale(perte).backward()
            scaler.step(optim)
            scaler.update()
            sched.step()
            somme += perte.item() * len(y)
            n += len(y)
        hist["perte"].append(somme / n)
        hist["prec_test"].append(evaluer(modele))
        hist["duree"].append(time.perf_counter() - t0)
        print(f"{nom} | époque {ep + 1}/{epoques} | perte {somme / n:.3f} | "
              f"test {hist['prec_test'][-1]:.1%} | {hist['duree'][-1]:.0f} s")
    return modele, hist

def nb_entraines(modele):
    return sum(p.numel() for p in modele.parameters() if p.requires_grad)

resultats = {}

## A. ResNet-50 de zéro

Poids aléatoires : le réseau doit tout apprendre (bords, textures, formes, races) à partir
de 3 680 images. C'est le point de comparaison.

In [ ]:
# 6. Expérience A : de zéro
seed_everything()
m_zero = models.resnet50(weights=None, num_classes=len(CLASSES))
print("Paramètres entraînés :", f"{nb_entraines(m_zero):,}".replace(",", " "))
m_zero, resultats["A. De zéro"] = entrainer(
    "A. De zéro", m_zero, [{"params": m_zero.parameters(), "lr": config["lr_zero"]}],
    config["epoques_zero"])
del m_zero
torch.cuda.empty_cache() if device == "cuda" else None

## B. Extraction de caractéristiques (tronc gelé)

On charge les poids appris sur ImageNet (1,28 million d'images, 1 000 classes), on **gèle**
tout le réseau et on remplace la dernière couche (2 048 → 1 000) par une couche neuve
(2 048 → 37). Seule celle-ci apprend : c'est une régression logistique sur les 2 048
caractéristiques que le tronc calcule pour chaque image.

In [ ]:
# 7. Expérience B : tronc gelé, nouvelle tête
seed_everything()
m_gel = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
for p in m_gel.parameters():
    p.requires_grad = False
m_gel.fc = nn.Linear(m_gel.fc.in_features, len(CLASSES))   # nouvelle couche, entraînable
print("Paramètres entraînés :", f"{nb_entraines(m_gel):,}".replace(",", " "))
m_gel, resultats["B. Tronc gelé"] = entrainer(
    "B. Tronc gelé", m_gel, [{"params": m_gel.fc.parameters(), "lr": config["lr_tete"]}],
    config["epoques_gel"], geler_bn=True)
del m_gel
torch.cuda.empty_cache() if device == "cuda" else None

## C. Réglage fin (fine-tuning) complet

Mêmes poids ImageNet, nouvelle tête, mais **tout** le réseau apprend : la tête avec un taux
de 10⁻³, le tronc avec un taux dix fois plus faible (10⁻⁴), pour ajuster les
caractéristiques sans effacer ce qu'ImageNet leur a appris.

In [ ]:
# 8. Expérience C : réglage fin de tout le réseau
seed_everything()
m_fine = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
m_fine.fc = nn.Linear(m_fine.fc.in_features, len(CLASSES))
tronc = [p for n, p in m_fine.named_parameters() if not n.startswith("fc.")]
print("Paramètres entraînés :", f"{nb_entraines(m_fine):,}".replace(",", " "))
m_fine, resultats["C. Réglage fin"] = entrainer(
    "C. Réglage fin", m_fine,
    [{"params": tronc, "lr": config["lr_tronc"]}, {"params": m_fine.fc.parameters(), "lr": config["lr_tete"]}],
    config["epoques_fine"])

In [ ]:
# 9. Bilan et courbes
print(f"{'expérience':<18} {'exactitude test':>15} {'durée':>9}")
for nom, h in resultats.items():
    print(f"{nom:<18} {h['prec_test'][-1]:>15.1%} {h['duree'][-1] / 60:>7.1f} min")

fig, ax = plt.subplots(figsize=(7, 4))
for nom, h in resultats.items():
    ax.plot(range(1, len(h["prec_test"]) + 1), [100 * v for v in h["prec_test"]], marker="o", label=nom)
ax.set(xlabel="époque", ylabel="exactitude en test (%)", ylim=(0, 100),
       title="Oxford-IIIT Pet (37 races) : ResNet-50")
ax.grid(alpha=0.3)
ax.legend()
fig.savefig("courbes_13b.png", dpi=150, bbox_inches="tight")
plt.show()

with open("resultats_13b.json", "w") as f:
    json.dump({"config": config, "device": device, "resultats": resultats}, f, ensure_ascii=False, indent=1)

# Seuils du README (vérifiés seulement sur la version complète)
if not TEST_LOCAL:
    a, b, c = (resultats[k]["prec_test"][-1] for k in ("A. De zéro", "B. Tronc gelé", "C. Réglage fin"))
    assert b >= 0.85, f"Tronc gelé : {b:.1%} (attendu ≥ 85 %)"
    assert c >= 0.88, f"Réglage fin : {c:.1%} (attendu ≥ 88 %)"
    assert a <= 0.50, f"De zéro : {a:.1%} (attendu ≤ 50 %)"
    print("Seuils atteints.")

## Grad-CAM sur le modèle réglé finement

Même méthode que dans l'exercice 13a, appliquée à la dernière couche de convolution de
ResNet-50 (`layer4`, cartes 2 048 × 7 × 7) : où le réseau a-t-il regardé pour décider de
la race ?

In [ ]:
# 10. Grad-CAM (chapitre 13) : poids d'une carte = moyenne de son gradient
def grad_cam(modele, x):
    modele.eval()
    cartes = {}
    crochet = modele.layer4.register_forward_hook(lambda m, e, s: cartes.__setitem__("A", s))
    logits = modele(x)
    crochet.remove()
    A = cartes["A"]
    c = logits.argmax(1)
    grad = torch.autograd.grad(logits.gather(1, c[:, None]).sum(), A)[0]
    cam = F.relu((grad.mean(dim=(2, 3), keepdim=True) * A).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-1], mode="bilinear", align_corners=False)[:, 0]
    cam = cam / cam.amax(dim=(1, 2), keepdim=True).clamp_min(1e-8)
    return cam.detach().float().cpu(), c.cpu(), logits.softmax(1).detach().float().cpu()

idx = list(range(0, len(brut_test), max(1, len(brut_test) // 6)))[:6]
x = torch.stack([trans_test(brut_test[i]["image"].convert("RGB")) for i in idx]).to(device)
cam, pred, proba = grad_cam(m_fine, x)
moy, et = torch.tensor(MOYENNE).view(3, 1, 1), torch.tensor(ECART_TYPE).view(3, 1, 1)
fig, axes = plt.subplots(2, len(idx), figsize=(2.2 * len(idx), 4.8), squeeze=False)
for k, i in enumerate(idx):
    img = (x[k].cpu() * et + moy).clamp(0, 1).permute(1, 2, 0).numpy()
    vrai = CLASSES[brut_test[i]["label"]]
    axes[0, k].imshow(img)
    axes[0, k].set_title(f"{CLASSES[pred[k]]} ({proba[k, pred[k]]:.0%})\nvrai : {vrai}", fontsize=8)
    axes[1, k].imshow(img)
    axes[1, k].imshow(cam[k], cmap="inferno", alpha=0.5)
    axes[0, k].axis("off")
    axes[1, k].axis("off")
fig.tight_layout()
fig.savefig("gradcam_13b.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 11. Sauvegarder les résultats AVANT la fin de la session (la machine Colab est effacée ensuite)
try:
    from google.colab import files
    for nom in ("resultats_13b.json", "courbes_13b.png", "gradcam_13b.png"):
        files.download(nom)
except ImportError:
    print("Hors Colab : fichiers écrits dans", os.getcwd())